# StateBridge × TwinAPI — Live Loopback HTTP Reproduction Notebook
Reproduces the **TwinAPI** Stateful False-Friend Audit and all **270 live loopback HTTP state-machine runs** (`Gemma-4-31B-IT` and `Gemma-4-26B-A4B-IT`) in `< 15 seconds` offline.

In [1]:
import json
from pathlib import Path

ROOT = Path(".").resolve()
if not (ROOT / "artifacts").exists():
  ROOT = ROOT.parent

ssot = json.loads((ROOT / "artifacts" / "metrics_ssot.json").read_text(encoding="utf-8"))
twin = ssot["twinapi_audit"]
print("=== TwinAPI Stateful False-Friend Audit (requests==2.34.2 vs httpx==0.28.1) ===")
print(f"Top-level function exact name overlap: {twin['top_level_exact_name_overlap_ratio']*100:.1f}% (8/9)")
print(f"Session/Client core verb overlap:      {twin['session_client_core_overlap_ratio']*100:.1f}% (10/12)")
print(f"Mean 256-d false-friend cosine sim:    {twin['mean_false_friend_embedding_cosine_256d']:.3f} ({twin['stateful_false_friend_count']}/{twin['total_pairs_audited']} pairs)")


=== TwinAPI Stateful False-Friend Audit (requests==2.34.2 vs httpx==0.28.1) ===
Top-level function exact name overlap: 88.9% (8/9)
Session/Client core verb overlap:      83.3% (10/12)
Mean 256-d false-friend cosine sim:    0.873 (12/15 pairs)


In [2]:
import sys
import time
SKILL_SCRIPTS = ROOT / "skills" / "state-bridge" / "scripts"
if str(SKILL_SCRIPTS) not in sys.path:
  sys.path.insert(0, str(SKILL_SCRIPTS))
import statebridge_core as sb

wfs = sb.get_benchmark_workflows()
t0 = time.monotonic()
sample_wf = wfs[0]
gold_res = sb.run_workflow_sandbox(sample_wf, sample_wf["httpx_gold_files"])
codemod_ctx = sb.build_arm_context(sample_wf, "arm1_codemod")
codemod_res = sb.run_workflow_sandbox(sample_wf, codemod_ctx["migrated_files"])
print(f"Loaded {len(wfs)} multi-step workflows across 5 divergence families.")
print(f"Sample {sample_wf['id']} Gold httpx outcome:    {gold_res['outcome_category']} (steps={gold_res['candidate_trace_len']})")
print(f"Sample {sample_wf['id']} Arm 1 Codemod outcome: {codemod_res['outcome_category']} at step={codemod_res['first_divergence_step']} ({codemod_res['divergence_field']})")
print(f"Full 270-run SSOT replay elapsed:      {ssot['benchmark']['replay_elapsed_sec']}s (< 15s requirement)")


Loaded 30 multi-step workflows across 5 divergence families.
Sample F1_01 Gold httpx outcome:    PASS_EQUIVALENT (steps=2)
Sample F1_01 Arm 1 Codemod outcome: PRE_HTTP_RUNTIME_ERROR at step=None (runtime_exception_before_http)
Full 270-run SSOT replay elapsed:      8.465s (< 15s requirement)


In [3]:
for model_key in ("gemma-4-31b-it", "gemma-4-26b-a4b-it"):
  m = ssot["models"][model_key]
  print(f"\n=== Model: {model_key} ({m['full_model_id']}) ===")
  for arm_id, arm_obj in m["arms"].items():
    rate_pct = arm_obj["solve_rate"] * 100
    print(f"  {arm_id:22s}: {rate_pct:5.1f}% ({arm_obj['pass_count']}/30) | prompt_tok={arm_obj['mean_prompt_tokens']} | thought_chars={arm_obj['mean_thought_chars']}")
  for cmp_key, cmp_obj in m["paired_comparisons"].items():
    print(f"  [{cmp_key}] diff=+{cmp_obj['diff']*100:.1f}% | discordant={cmp_obj['discordant_a_wins']}:{cmp_obj['discordant_b_wins']} | McNemar p={cmp_obj['mcnemar_p']} | 95% CI=[{cmp_obj['ci95_low']*100:.1f}%, {cmp_obj['ci95_high']*100:.1f}%]")



=== Model: gemma-4-31b-it (google/models/gemma-4-31b-it) ===
  arm1_codemod          :   0.0% (0/30) | prompt_tok=0 | thought_chars=0
  arm2_target_only      :  13.3% (4/30) | prompt_tok=497.7 | thought_chars=4572.2
  arm3_embedding_topk   :  13.3% (4/30) | prompt_tok=637.7 | thought_chars=4764.6
  arm4_callgraph_slice  :  90.0% (27/30) | prompt_tok=799.9 | thought_chars=8396.0
  arm5_unrelated_slice  :  16.7% (5/30) | prompt_tok=702.7 | thought_chars=5647.3
  [arm4_vs_arm1] diff=+90.0% | discordant=27:0 | McNemar p=1e-08 | 95% CI=[76.7%, 100.0%]
  [arm4_vs_arm2] diff=+76.7% | discordant=23:0 | McNemar p=2.4e-07 | 95% CI=[60.0%, 90.0%]
  [arm4_vs_arm3] diff=+76.7% | discordant=23:0 | McNemar p=2.4e-07 | 95% CI=[60.0%, 90.0%]
  [arm4_vs_arm5] diff=+73.3% | discordant=23:1 | McNemar p=2.98e-06 | 95% CI=[53.3%, 90.0%]

=== Model: gemma-4-26b-a4b-it (google/models/gemma-4-26b-a4b-it) ===
  arm1_codemod          :   0.0% (0/30) | prompt_tok=0 | thought_chars=0
  arm2_target_only      :  13

In [4]:
runs = [json.loads(line) for line in (ROOT / "artifacts" / "runs.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
print(f"Total evaluated live loopback HTTP runs in runs.jsonl: {len(runs)}")
for fam in ("F1_REDIRECT_FOLLOW_DEFAULT", "F2_CROSS_ORIGIN_AUTH_STRIP", "F3_COOKIE_JAR_SCOPING", "F4_STREAM_CONTEXT_LIFECYCLE", "F5_CLIENT_INIT_AND_STATUS_ERROR"):
  p31 = ssot["models"]["gemma-4-31b-it"]["arms"]["arm4_callgraph_slice"]["by_family"][fam]["pass_count"]
  p26 = ssot["models"]["gemma-4-26b-a4b-it"]["arms"]["arm4_callgraph_slice"]["by_family"][fam]["pass_count"]
  print(f"  {fam:34s} | 31B-IT Arm 4: {p31}/6 | 26B-A4B-IT Arm 4: {p26}/6")


Total evaluated live loopback HTTP runs in runs.jsonl: 270
  F1_REDIRECT_FOLLOW_DEFAULT         | 31B-IT Arm 4: 6/6 | 26B-A4B-IT Arm 4: 6/6
  F2_CROSS_ORIGIN_AUTH_STRIP         | 31B-IT Arm 4: 6/6 | 26B-A4B-IT Arm 4: 5/6
  F3_COOKIE_JAR_SCOPING              | 31B-IT Arm 4: 5/6 | 26B-A4B-IT Arm 4: 5/6
  F4_STREAM_CONTEXT_LIFECYCLE        | 31B-IT Arm 4: 6/6 | 26B-A4B-IT Arm 4: 6/6
  F5_CLIENT_INIT_AND_STATUS_ERROR    | 31B-IT Arm 4: 4/6 | 26B-A4B-IT Arm 4: 6/6
